In [1]:
# %% [markdown]
# # 05b_full_run_diagnostic.ipynb
# Diagnoses the full 300-record v2 run's compile_error and
# wrong_result cases: which questions have ALL 5 iterations failing
# the same way (a systemic, fixable pattern) vs. intermittent failures
# (likely LLM variance, less fixable via prompt/YAML changes).

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import pandas as pd
pd.set_option('display.max_colwidth', 150)

df = pd.read_csv('results_phase3_v2_arch.csv')
print(f"Loaded {len(df)} rows")

# %% [markdown]
# ## 1. Which questions have 5/5 compile errors (systemic failure)?

# %%
compile_err_by_q = df.groupby('question_id').apply(
    lambda g: pd.Series({
        'n': len(g),
        'compile_err': g['compile_error'].notna().sum(),
        'exec_err': g['execution_error'].notna().sum(),
        'correct': g['is_correct'].sum(),
    })
)
# drop the now-redundant question_id column if pandas' older
# behavior (pre-2.2, no include_groups param) included it
if 'question_id' in compile_err_by_q.columns:
    compile_err_by_q = compile_err_by_q.drop(columns=['question_id'])

all_compile_fail = compile_err_by_q[compile_err_by_q['compile_err'] == 5]
print(f"Questions with 5/5 compile_error (fully blocked by a systemic issue): {len(all_compile_fail)}")
print(all_compile_fail)

# %% [markdown]
# ## 2. Distinct compile_error MESSAGES for those fully-blocked
#     questions -- group by error type to see if a small number of
#     root causes explain most of the 28.3% compile error rate

# %%
print("\n" + "=" * 70)
print("COMPILE ERROR MESSAGES for fully-blocked questions")
print("=" * 70)

for qid in all_compile_fail.index:
    sub = df[df['question_id'] == qid]
    print(f"\n[{qid}] category={sub['category'].iloc[0]}, difficulty={sub['difficulty'].iloc[0]}")
    # show the first iteration's full detail as representative
    row = sub[sub['compile_error'].notna()].iloc[0]
    print(f"  concept_sql: {row['concept_sql']}")
    print(f"  ERROR: {row['compile_error']}")

# %% [markdown]
# ## 3. Categorize ALL compile errors (not just fully-blocked
#     questions) by error TYPE, to find the highest-leverage fix

# %%
print("\n" + "=" * 70)
print("COMPILE ERROR TYPE BREAKDOWN (all 85 compile errors)")
print("=" * 70)

df_compile_err = df[df['compile_error'].notna()].copy()

def classify_error(err):
    if 'WHERE clause' in err:
        return 'measure-in-WHERE (WHERE-guard catch)'
    elif 'MISMATCHED' in err or 'join hint' in err:
        return 'join hint mismatch/unresolved'
    elif 'Excluded concept' in err:
        return 'excluded concept used (status_label etc.)'
    else:
        return 'other/unclassified'

df_compile_err['error_type'] = df_compile_err['compile_error'].apply(classify_error)
print(df_compile_err['error_type'].value_counts())

print("\nBreakdown by category:")
print(df_compile_err.groupby(['category', 'error_type']).size().unstack(fill_value=0))

# %% [markdown]
# ## 4. Wrong-result cases: which questions, and are they clustered
#     around known limitations (e.g. measures like avg_amount that
#     don't support multi-step resegmentation)?

# %%
print("\n" + "=" * 70)
print("WRONG-RESULT QUESTIONS (ran successfully, answer didn't match gold)")
print("=" * 70)

df['is_wrong_result'] = (
    df['compile_error'].isna() &
    df['execution_error'].isna() &
    (df['is_correct'] == False)
)

wrong_by_q = df[df['is_wrong_result']].groupby('question_id').size().sort_values(ascending=False)
print(f"\nQuestions with at least one wrong_result iteration: {len(wrong_by_q)}")
print(wrong_by_q)

print("\n" + "=" * 70)
print("Sample detail for the top 5 most-frequently-wrong questions")
print("=" * 70)
for qid in wrong_by_q.head(5).index:
    sub = df[(df['question_id'] == qid) & df['is_wrong_result']]
    row = sub.iloc[0]
    print(f"\n[{qid}] category={row['category']}, difficulty={row['difficulty']} "
          f"({len(sub)}/5 iterations wrong)")
    print(f"  concept_sql : {row['concept_sql']}")
    print(f"  compiled_sql: {row['compiled_sql']}")
    print(f"  gold_sql    : {row['gold_sql']}")

# %% [markdown]
# ## 5. Execution error detail -- how many are TIMEOUTS specifically
#     (a measurement artifact / query complexity issue) vs. genuine
#     SQL errors (a compiler or logic issue)?

# %%
print("\n" + "=" * 70)
print("EXECUTION ERROR TYPE BREAKDOWN")
print("=" * 70)

df_exec_err = df[df['execution_error'].notna()].copy()
df_exec_err['is_timeout'] = df_exec_err['execution_error'].str.contains('timeout', case=False, na=False)

print(f"Total execution errors: {len(df_exec_err)}")
print(f"  Timeouts: {df_exec_err['is_timeout'].sum()}")
print(f"  Other SQL errors: {(~df_exec_err['is_timeout']).sum()}")

if df_exec_err['is_timeout'].sum() > 0:
    print("\nQuestions with timeout errors:")
    print(df_exec_err[df_exec_err['is_timeout']].groupby('question_id').size())

print("\nNon-timeout execution error messages (sample):")
non_timeout = df_exec_err[~df_exec_err['is_timeout']]
for qid in non_timeout['question_id'].unique()[:10]:
    row = non_timeout[non_timeout['question_id'] == qid].iloc[0]
    print(f"\n[{qid}]: {row['execution_error'][:150]}")

# %% [markdown]
# ## 6. Summary: how much of the 33.7% overall accuracy gap (vs v1's
#     38.3%) is explained by each failure category?

# %%
print("\n" + "=" * 70)
print("FAILURE BREAKDOWN SUMMARY")
print("=" * 70)
total = len(df)
n_correct = df['is_correct'].sum()
n_compile = df['compile_error'].notna().sum()
n_exec = df['execution_error'].notna().sum()
n_timeout = df_exec_err['is_timeout'].sum() if len(df_exec_err) > 0 else 0
n_exec_real = n_exec - n_timeout
n_wrong = df['is_wrong_result'].sum()

print(f"Total records: {total}")
print(f"  Correct:                    {n_correct:3d} ({n_correct/total:.1%})")
print(f"  Compile errors:             {n_compile:3d} ({n_compile/total:.1%})")
print(f"  Execution errors (timeout): {n_timeout:3d} ({n_timeout/total:.1%})")
print(f"  Execution errors (other):   {n_exec_real:3d} ({n_exec_real/total:.1%})")
print(f"  Wrong result:               {n_wrong:3d} ({n_wrong/total:.1%})")

print("""

INTERPRETATION:
- High compile_error concentrated in a FEW question_ids with 5/5
  failure -> targeted few-shot fix, likely worth another small
  prompt iteration before finalizing.
- Compile errors spread thinly across MANY questions with 1-3/5
  failure -> likely LLM sampling variance on borderline cases, less
  fixable via prompt changes alone; consider reporting as-is with
  the compile-vs-execution taxonomy documented in the manuscript.
- Timeouts should be reported SEPARATELY from "genuine" execution
  errors in the manuscript's failure taxonomy (per R1 Concern #7's
  request for a complete failure taxonomy: correct / incorrect-
  executable / execution failure / timeout / degenerate).
- Wrong-result questions clustered on measures with known scope
  limits (e.g. avg_amount not supporting monthly resegmentation)
  should be documented as a design limitation, not chased further
  with prompt engineering.
""")

SCRIPT VERSION: 2026-07-15-v1
Loaded 300 rows
Questions with 5/5 compile_error (fully blocked by a systemic issue): 5
             n  compile_err  exec_err  correct
question_id                                   
CR16         5            5         0        0
CR28         5            5         0        0
CR34         5            5         0        0
CR35         5            5         0        0
CR42         5            5         0        0

COMPILE ERROR MESSAGES for fully-blocked questions

[CR16] category=transaction_behavior, difficulty=moderate
  concept_sql: SELECT COUNT(DISTINCT account.account_id)
   FROM account
   JOIN trans ON trans.account_link
   WHERE trans.avg_balance < 0
  ERROR: Invalid SQL: an aggregate function (from a measure concept) was used directly inside a WHERE clause. Measures are pre-aggregated and can only be used in SELECT, HAVING, or ORDER BY -- not WHERE, since WHERE filters rows before aggregation. If you need a row-level existence check (e.g. 'has at

C:\Users\miy\AppData\Local\Temp\ipykernel_30680\2333478181.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  compile_err_by_q = df.groupby('question_id').apply(


In [2]:
"""
measure-in-WHERE 79건 재분류 스크립트
- API 재호출 없음. results_phase2_bird_v2_arch.csv만 읽음.
- sqlglot으로 concept_sql을 파싱해서, WHERE 절 안에 등장하는 집계함수가
  "최상위(top-level) WHERE에 직접" 있는지, 아니면 서브쿼리 안이거나
  GROUP BY와 함께 쓰인 정당한 패턴인지를 구조적으로 구분한다.
- 정규식/문자열 매칭이 아니라 AST 기반 판정이므로, 116번 같은
  "서브쿼리 안에서 GROUP BY와 함께 AVG를 쓴" 케이스를 오탐으로 잡아낼 수 있다.

필요 패키지: pip install sqlglot
"""

import pandas as pd
import sqlglot
from sqlglot import exp

INPUT_CSV = "results_phase2_bird_v2_arch.csv"
OUTPUT_CSV = "measure_in_where_reclassified.csv"

AGGREGATE_FUNCS = {
    "AVG", "SUM", "COUNT", "MIN", "MAX", "GROUP_CONCAT",
    "STDEV", "VARIANCE", "TOTAL",
}


def find_top_level_where_aggregate_violations(sql: str):
    """
    concept_sql을 파싱해서, 각 SELECT 문(서브쿼리 포함)마다:
      - 그 SELECT 자신의 WHERE 절 안에
      - 그 SELECT 자신의 FROM/JOIN에 속한 컬럼을 인자로 하는 집계함수가
      - 직접(다른 서브쿼리에 감싸이지 않고) 등장하는지를 확인한다.

    반환: 위반이 발견되면 위반 상세 리스트, 없으면 빈 리스트.
    각 항목은 {"select_sql": str, "agg_func": str, "agg_sql": str} 형태.
    """
    violations = []
    try:
        tree = sqlglot.parse_one(sql, read="sqlite")
    except Exception as e:
        return [{"select_sql": None, "agg_func": "PARSE_ERROR", "agg_sql": str(e)}]

    # 트리 안의 모든 SELECT 노드를 순회 (서브쿼리 포함, 중첩 깊이 무관)
    for select_node in tree.find_all(exp.Select):
        where_clause = select_node.args.get("where")
        if where_clause is None:
            continue

        # 이 WHERE절 하위에서 집계함수(AggFunc)를 찾되,
        # 그 사이에 다른 Select(서브쿼리)가 끼어있으면 그건 "이 SELECT의 것"이 아니므로 제외.
        for agg_node in where_clause.find_all(exp.AggFunc):
            # agg_node로부터 부모로 거슬러 올라가며, where_clause에 도달하기 전에
            # 또 다른 Select를 만나면 -> 그 집계는 서브쿼리 내부 것이므로 정당함(스킵)
            is_nested_in_subquery = False
            parent = agg_node.parent
            while parent is not None and parent is not where_clause:
                if isinstance(parent, exp.Select):
                    is_nested_in_subquery = True
                    break
                parent = parent.parent

            if is_nested_in_subquery:
                continue  # 서브쿼리 안의 집계 -> 정당, 위반 아님

            func_name = agg_node.sql_name().upper() if hasattr(agg_node, "sql_name") else type(agg_node).__name__.upper()
            violations.append({
                "select_sql": select_node.sql(dialect="sqlite"),
                "agg_func": func_name,
                "agg_sql": agg_node.sql(dialect="sqlite"),
            })

    return violations


def classify_row(concept_sql: str) -> dict:
    """단일 행 판정. 반환 dict에 재분류 결과와 근거를 담는다."""
    if not isinstance(concept_sql, str) or not concept_sql.strip():
        return {
            "reclass_verdict": "NO_SQL_TEXT",
            "n_violations_found": 0,
            "violation_detail": "",
            "parse_ok": False,
        }

    violations = find_top_level_where_aggregate_violations(concept_sql)

    if violations and violations[0]["agg_func"] == "PARSE_ERROR":
        return {
            "reclass_verdict": "PARSE_FAILED_MANUAL_REVIEW_NEEDED",
            "n_violations_found": 0,
            "violation_detail": violations[0]["agg_sql"],
            "parse_ok": False,
        }

    if len(violations) == 0:
        verdict = "FALSE_POSITIVE_LIKELY"  # 컴파일러가 오탐했을 가능성이 높음
    else:
        verdict = "TRUE_VIOLATION_CONFIRMED"  # 진짜로 top-level WHERE에 집계함수가 있음

    detail = " | ".join(
        f"{v['agg_func']} in: {v['agg_sql']}" for v in violations
    )

    return {
        "reclass_verdict": verdict,
        "n_violations_found": len(violations),
        "violation_detail": detail,
        "parse_ok": True,
    }


def main():
    df = pd.read_csv(INPUT_CSV)

    print(f"Loaded {len(df)} total records from {INPUT_CSV}")
    print(f"Columns available: {list(df.columns)}\n")

    # measure-in-WHERE로 표시된 행만 필터링
    # (compile_error 컬럼에 "WHERE clause" 문구가 포함된 행 -- 05번 노트북 기준)
    mask = df["compile_error"].astype(str).str.contains("WHERE clause", na=False)
    subset = df[mask].copy()

    print(f"{len(subset)} rows flagged as measure-in-WHERE failures.\n")

    if len(subset) == 0:
        print("No matching rows found. Check the compile_error column contents:")
        print(df["compile_error"].dropna().unique()[:20])
        return

    # concept_sql 컬럼이 없을 경우를 대비해 확인
    sql_col = "concept_sql" if "concept_sql" in df.columns else None
    if sql_col is None:
        possible = [c for c in df.columns if "sql" in c.lower()]
        print(f"!! 'concept_sql' column not found. Candidates: {possible}")
        print("!! Edit sql_col below to match your actual column name, then rerun.")
        return

    results = subset[sql_col].apply(classify_row).apply(pd.Series)
    subset = pd.concat([subset.reset_index(drop=True), results.reset_index(drop=True)], axis=1)

    # ---- 요약 출력 ----
    print("=" * 70)
    print("RECLASSIFICATION SUMMARY")
    print("=" * 70)
    verdict_counts = subset["reclass_verdict"].value_counts()
    print(verdict_counts.to_string())
    print()

    n_total = len(subset)
    n_false_pos = (subset["reclass_verdict"] == "FALSE_POSITIVE_LIKELY").sum()
    n_true_violation = (subset["reclass_verdict"] == "TRUE_VIOLATION_CONFIRMED").sum()
    n_needs_manual = n_total - n_false_pos - n_true_violation

    print(f"Of {n_total} flagged rows:")
    print(f"  {n_true_violation} ({n_true_violation/n_total:.1%}) confirmed as genuine top-level "
          f"WHERE-clause aggregate violations")
    print(f"  {n_false_pos} ({n_false_pos/n_total:.1%}) appear to be compiler false positives "
          f"(aggregate is inside a subquery or paired with GROUP BY, not a top-level WHERE misuse)")
    print(f"  {n_needs_manual} ({n_needs_manual/n_total:.1%}) could not be parsed automatically "
          f"and need manual review")
    print()

    # 오탐으로 분류된 것들 중 몇 개를 바로 검토할 수 있게 출력
    false_positives = subset[subset["reclass_verdict"] == "FALSE_POSITIVE_LIKELY"]
    print("=" * 70)
    print(f"SAMPLE FALSE POSITIVES (up to 10 of {len(false_positives)})")
    print("=" * 70)
    for _, row in false_positives.head(10).iterrows():
        qid = row.get("question_id", "?")
        print(f"\n--- question_id={qid} ---")
        print(f"question: {row.get('question', '')}")
        print(f"concept_sql:\n{row[sql_col]}")
        print(f"(compiler said: {str(row['compile_error'])[:150]})")

    # 수동 검토가 필요한 것들도 출력
    manual = subset[subset["reclass_verdict"] == "PARSE_FAILED_MANUAL_REVIEW_NEEDED"]
    if len(manual) > 0:
        print("\n" + "=" * 70)
        print(f"PARSE FAILURES NEEDING MANUAL REVIEW ({len(manual)})")
        print("=" * 70)
        for _, row in manual.iterrows():
            qid = row.get("question_id", "?")
            print(f"\n--- question_id={qid} ---")
            print(f"parse error: {row['violation_detail']}")
            print(f"concept_sql:\n{row[sql_col]}")

    # ---- 저장 ----
    subset.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved full reclassification (with per-row verdict and detail) to: {OUTPUT_CSV}")
    print("Use this CSV to spot-check TRUE_VIOLATION_CONFIRMED and FALSE_POSITIVE_LIKELY")
    print("rows manually before citing the corrected count in the manuscript.")


if __name__ == "__main__":
    main()

Loaded 530 total records from results_phase2_bird_v2_arch.csv
Columns available: ['question_id', 'difficulty', 'category', 'iteration', 'is_correct', 'compile_error', 'execution_error', 'concept_sql', 'compiled_sql', 'gold_sql', 'question', 'latency_ms', 'prompt_chars', 'substitutions']

79 rows flagged as measure-in-WHERE failures.

RECLASSIFICATION SUMMARY
reclass_verdict
FALSE_POSITIVE_LIKELY    79

Of 79 flagged rows:
  0 (0.0%) confirmed as genuine top-level WHERE-clause aggregate violations
  79 (100.0%) appear to be compiler false positives (aggregate is inside a subquery or paired with GROUP BY, not a top-level WHERE misuse)
  0 (0.0%) could not be parsed automatically and need manual review

SAMPLE FALSE POSITIVES (up to 10 of 79)

--- question_id=95 ---
question: List out the account numbers of clients who are youngest and have highest average salary?
concept_sql:
SELECT account.account_id
   FROM account
   JOIN disp ON account.disp_link
   JOIN client ON disp.client_link
  

In [3]:
"""
measure-in-WHERE 79건 재분류 스크립트 v2
- v1의 결함: exp.AggFunc(즉 진짜 AVG()/SUM() 함수 호출) 유무만 봤음.
  실제로는 semantic layer의 "measure"가 컴파일된 SQL에서 이미
  별칭이 붙은 컬럼명(예: total_amount, avg_salary)으로 나타날 수 있어서,
  함수 호출 형태가 없어도 여전히 규칙 위반일 수 있다.
- v2는 YAML에서 measure 이름 목록을 읽어와서, 그 이름이
  "최상위 WHERE 절 안에" 등장하는지를 직접 확인한다.
  (서브쿼리 안의 WHERE/GROUP BY/ORDER BY는 별도로 취급)

필요 패키지: pip install sqlglot pyyaml
"""

import re
import pandas as pd
import yaml
import sqlglot
from sqlglot import exp

INPUT_CSV = "results_phase2_bird_v2_arch.csv"
YAML_PATH = "financial_semantic_layer.yaml"
OUTPUT_CSV = "measure_in_where_reclassified_v2.csv"


def load_measure_names(yaml_path: str) -> set:
    """
    YAML에서 정의된 모든 measure 이름을 모은다.
    financial_semantic_layer.yaml의 실제 구조를 모르므로, 최대한 관대하게
    'measures:' 키 아래에 있는 항목의 name을 전부 수집한다.
    구조가 다르면 print된 raw YAML 일부를 보고 파싱 부분만 조정하면 된다.
    """
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    names = set()

    def walk(node):
        if isinstance(node, dict):
            if "measures" in node and isinstance(node["measures"], list):
                for m in node["measures"]:
                    if isinstance(m, dict) and "name" in m:
                        names.add(m["name"])
            for v in node.values():
                walk(v)
        elif isinstance(node, list):
            for item in node:
                walk(item)

    walk(data)
    return names


def find_top_level_where(sql: str, measure_names: set):
    """
    각 SELECT 노드(서브쿼리 포함)의 '자기 자신의' WHERE 절 텍스트 안에서,
    measure_names에 있는 이름이 컬럼 참조(Column)로 등장하는지 확인한다.
    서브쿼리 내부의 WHERE는 그 서브쿼리 자신의 최상위 WHERE로 별도 취급되므로,
    "이 WHERE가 통째로 다른 SELECT 안에 있는가"는 문제 삼지 않는다 --
    컴파일러 규칙 자체가 "어떤 WHERE든 measure를 직접 넣으면 안 된다"이므로,
    서브쿼리의 WHERE라도 measure가 직접 있으면 위반이다.
    (GROUP BY/ORDER BY/HAVING/SELECT 절에 있는 것은 위반이 아니다.)
    """
    violations = []
    try:
        tree = sqlglot.parse_one(sql, read="sqlite")
    except Exception as e:
        return None, str(e)

    for select_node in tree.find_all(exp.Select):
        where_clause = select_node.args.get("where")
        if where_clause is None:
            continue

        for col_node in where_clause.find_all(exp.Column):
            col_name = col_node.name  # 예: 'total_amount', 'avg_salary'
            if col_name in measure_names:
                violations.append({
                    "measure_name": col_name,
                    "where_sql": where_clause.sql(dialect="sqlite"),
                    "select_sql": select_node.sql(dialect="sqlite"),
                })

    return violations, None


def classify_row(concept_sql: str, measure_names: set) -> dict:
    if not isinstance(concept_sql, str) or not concept_sql.strip():
        return {
            "reclass_verdict_v2": "NO_SQL_TEXT",
            "n_violations_v2": 0,
            "violation_detail_v2": "",
        }

    violations, err = find_top_level_where(concept_sql, measure_names)

    if err is not None:
        return {
            "reclass_verdict_v2": "PARSE_FAILED_MANUAL_REVIEW_NEEDED",
            "n_violations_v2": 0,
            "violation_detail_v2": err,
        }

    if len(violations) == 0:
        verdict = "FALSE_POSITIVE_LIKELY"
        detail = ""
    else:
        verdict = "TRUE_VIOLATION_CONFIRMED"
        detail = " | ".join(
            f"measure '{v['measure_name']}' in WHERE: {v['where_sql']}" for v in violations
        )

    return {
        "reclass_verdict_v2": verdict,
        "n_violations_v2": len(violations),
        "violation_detail_v2": detail,
    }


def main():
    measure_names = load_measure_names(YAML_PATH)
    print(f"Loaded {len(measure_names)} measure names from {YAML_PATH}:")
    print(sorted(measure_names))
    print()

    if len(measure_names) == 0:
        print("!! No measure names found. The YAML structure may differ from")
        print("!! what load_measure_names() expects. Printing raw top-level keys:")
        with open(YAML_PATH, "r", encoding="utf-8") as f:
            raw = yaml.safe_load(f)
        print(list(raw.keys()) if isinstance(raw, dict) else type(raw))
        print("Inspect financial_semantic_layer.yaml manually and adjust")
        print("load_measure_names() to match its actual nesting.")
        return

    df = pd.read_csv(INPUT_CSV)
    mask = df["compile_error"].astype(str).str.contains("WHERE clause", na=False)
    subset = df[mask].copy()
    print(f"{len(subset)} rows flagged as measure-in-WHERE failures.\n")

    results = subset["concept_sql"].apply(lambda s: classify_row(s, measure_names)).apply(pd.Series)
    subset = pd.concat([subset.reset_index(drop=True), results.reset_index(drop=True)], axis=1)

    print("=" * 70)
    print("RECLASSIFICATION SUMMARY (v2 -- semantic/measure-name based)")
    print("=" * 70)
    print(subset["reclass_verdict_v2"].value_counts().to_string())
    print()

    n_total = len(subset)
    n_true = (subset["reclass_verdict_v2"] == "TRUE_VIOLATION_CONFIRMED").sum()
    n_false = (subset["reclass_verdict_v2"] == "FALSE_POSITIVE_LIKELY").sum()
    n_manual = n_total - n_true - n_false

    print(f"Of {n_total} flagged rows:")
    print(f"  {n_true} ({n_true/n_total:.1%}) confirmed genuine (measure name appears directly in a WHERE clause)")
    print(f"  {n_false} ({n_false/n_total:.1%}) likely compiler false positives")
    print(f"  {n_manual} ({n_manual/n_total:.1%}) need manual review (parse failure)")
    print()

    # 오탐 후보 샘플 출력 -- 이번엔 진짜로 WHERE에 measure 이름이 없는 것들만 나와야 함
    false_positives = subset[subset["reclass_verdict_v2"] == "FALSE_POSITIVE_LIKELY"]
    print("=" * 70)
    print(f"SAMPLE FALSE POSITIVES (up to 10 of {len(false_positives)})")
    print("=" * 70)
    for _, row in false_positives.head(10).iterrows():
        print(f"\n--- question_id={row.get('question_id', '?')} ---")
        print(f"question: {row.get('question', '')}")
        print(f"concept_sql:\n{row['concept_sql']}")

    print("\n" + "=" * 70)
    print(f"SAMPLE TRUE VIOLATIONS (up to 10 of {n_true})")
    print("=" * 70)
    true_violations = subset[subset["reclass_verdict_v2"] == "TRUE_VIOLATION_CONFIRMED"]
    for _, row in true_violations.head(10).iterrows():
        print(f"\n--- question_id={row.get('question_id', '?')} ---")
        print(f"question: {row.get('question', '')}")
        print(f"detail: {row['violation_detail_v2']}")

    subset.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


if __name__ == "__main__":
    main()

Loaded 12 measure names from financial_semantic_layer.yaml:
['avg_amount', 'avg_balance', 'avg_salary_mean', 'count', 'default_count', 'default_rate', 'female_count', 'gold_count', 'male_count', 'owner_count', 'running_count', 'total_amount']

79 rows flagged as measure-in-WHERE failures.

RECLASSIFICATION SUMMARY (v2 -- semantic/measure-name based)
reclass_verdict_v2
TRUE_VIOLATION_CONFIRMED    61
FALSE_POSITIVE_LIKELY       18

Of 79 flagged rows:
  61 (77.2%) confirmed genuine (measure name appears directly in a WHERE clause)
  18 (22.8%) likely compiler false positives
  0 (0.0%) need manual review (parse failure)

SAMPLE FALSE POSITIVES (up to 10 of 18)

--- question_id=95 ---
question: List out the account numbers of clients who are youngest and have highest average salary?
concept_sql:
SELECT account.account_id
   FROM account
   JOIN disp ON account.disp_link
   JOIN client ON disp.client_link
   JOIN district ON client.district_link
   WHERE client.birth_date = (SELECT MIN(cli

In [4]:
df = pd.read_csv("measure_in_where_reclassified_v2.csv")
fp = df[df["reclass_verdict_v2"] == "FALSE_POSITIVE_LIKELY"]
print(fp["question_id"].nunique(), "unique questions among", len(fp), "records")
print(sorted(fp["question_id"].unique()))

7 unique questions among 18 records
[95, 115, 116, 133, 134, 145, 155]
